In [1]:
# Open pointcloud file 
from pathlib import Path
# This is for the meantime -------------------------------------------------------
# The final motivation is to make it pip installeable 
import sys
src_path = Path.cwd().parents[1]
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
# --------------------------------------------------------------------------------
from attpc_analysis.src.load.io import RunPointCloudDataLoader

In [28]:
workspace_path = Path("/mnt/analysis/argonne2023/ramirezc/a2066/workspace_369_trace/")
# Load class with general details of the runs
empty_files = []
cloud_data = RunPointCloudDataLoader(
    pointcloud_result_path= workspace_path / f"Pointcloud"
)


# load the specific runs and events 
import polars as pl 
import numpy as np 
check_events = pl.read_csv("./../../personal_analysis/tail_elastic_dent.csv") # with noise
runs = check_events["run"]

export_point_cloud = {}
export_point_cloud["run"] = np.empty(0,int)
export_point_cloud["event"] = np.empty(0,int)
export_point_cloud["x"] = np.empty(0,float)
export_point_cloud["y"] = np.empty(0,float)
export_point_cloud["z"] = np.empty(0,float)
export_point_cloud["charge"] = np.empty(0,float)


import csv


file_number = 0
counter = 1


for run in runs:
    mask = check_events["run"] == run 
    events = check_events.select("event").to_numpy()[mask]
    for event in events: 
        event = int(event)
        cloud = cloud_data.load_pointcloud(run,event)
        if cloud is None: continue 
        filename = f"cloud_deuteron_{file_number}.csv"
        with open(filename, "a", newline="", encoding="utf-8" ) as file: 
            writer = csv.writer(file)
            header = ["Run",run,"Event",event]
            writer.writerow(header)
            subheader = ["x","y","z","charge"]
            writer.writerow(subheader)
            for row in range(len(cloud.data[:,0])):
                content = cloud.data[row,:3].tolist()
                content.append(cloud.data[row,4])
                writer.writerow(content)

            counter+=1
            if counter == 100: 
                counter = 1
                file_number +=1

/tmp/ipykernel_28377/2859989172.py:35: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  event = int(event)


In [16]:
cloud.data[0,:3].tolist()

[-34.357647, -216.78348, 10.244592025819065]